# 도서 인기도 베이스라인(Popularity Baseline) 산출

`data/raw/top-list`의 교보문고 종합 베스트셀러 상품리스트 xlsx 파일 48개는 서로 겹치지 않는 월 단위 기간의 스냅샷이다 (같은 달에 두 파일이 존재하지 않음). 이 파일들을 모두 읽어
1. 완전히 동일한 파일(내용 중복)이 있는지 확인하고, 있다면 제외하고
2. 파일 내 순위와 여러 파일에 걸쳐 등장한 횟수(=몇 개월간 리스트에 남아있었는지)를 모두 반영한 점수로 최종 인기도 순위를 매겨
3. `book_popularity_baseline.csv`로 저장한다.

## 점수 공식과 근거

**1차 시도** — 역순위 합산(reciprocal rank sum):
`raw_score(도서) = Σ (1 / 해당 파일에서의 순위)`

이 방식은 등장 횟수가 늘어날수록 더할 항이 그대로 늘어나기 때문에, 사실상 중복횟수에 대해 **선형(linear)** 으로 가중치가 붙는다. 실제로 확인해보니 순위 3~6위(최상위권)를 찍었지만 2~6개월만 리스트에 있었던 책들이, 순위는 평범(12위 안팎)해도 48개월 내내 리스트에 남아있던 책보다 훨씬 낮은 점수를 받는 역전 현상이 다수 발생했다 — 즉 '몇 달이나 리스트에 남아있었는가'가 '얼마나 최상위였는가'를 과도하게 압도하는 문제가 있었다.

**최종 채택** — 등장 횟수에 제곱근 감쇠(diminishing return)를 적용:
`score(도서) = raw_score / sqrt(중복횟수) = 평균순위점수 × sqrt(중복횟수)`

- `raw_score / 중복횟수` = 등장할 때마다의 평균 순위 품질 (한 번 등장해도 반영되는 핵심 신호)
- `sqrt(중복횟수)` = 오래 리스트에 남아있을수록 보너스를 주되, 늘어날수록 한계 기여도가 줄어드는 감쇠 계수 (48개월과 2개월의 차이는 24배가 아니라 √24 ≈ 4.9배 정도로만 반영)

제곱근 감쇠는 정보검색 분야에서 등장 빈도(TF)를 그대로 쓰지 않고 `sqrt(tf)`나 `log(1+tf)`로 완화해 쓰는 것과 같은 발상이다 — 빈도가 의미 있는 신호이긴 하지만 그대로 선형으로 누적시키면 '단순히 오래 노출된 것'과 '실제로 인기 있는 것'을 구분하지 못하게 되기 때문이다.

In [1]:
import glob
import hashlib
import warnings
from pathlib import Path

import pandas as pd

# openpyxl이 스타일 정보가 없는 워크북에 대해 매 파일마다 띄우는 경고를 숨긴다
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

INPUT_DIR = "../data/raw/top-list"
FILE_PATTERN = "교보문고_종합_베스트셀러_상품리스트*.xlsx"

KEY_COLUMN = "상품코드"  # ISBN 기준으로 동일 도서 판단
RANK_COLUMN = "순위"

# 실제 도서가 아닌 굿즈 세트 등 순위 산출에서 제외할 상품명
EXCLUDED_TITLES = {"데뷔 못 하면 죽는 병 걸림 4부 초판 한정 굿즈박스 세트"}

OUTPUT_PATH = "../data/processed/book_popularity_baseline.csv"


In [2]:
files = sorted(glob.glob(f"{INPUT_DIR}/{FILE_PATTERN}"))
print(f"발견된 파일 {len(files)}개")
for f in files:
    print(" -", f)


발견된 파일 48개
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (15).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (16).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (17).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (18).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (19).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (20).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (21).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (22).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (23).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (24).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (25).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (26).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (27).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (28).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (29).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (30).xlsx
 - ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (31).xlsx
 - ../data/raw/to

In [3]:
# 완전히 동일한 파일(바이트 단위)이 있는지 확인하고, 있다면 각 그룹의 첫 파일만 사용한다
hash_to_files = {}
for f in files:
    digest = hashlib.md5(Path(f).read_bytes()).hexdigest()
    hash_to_files.setdefault(digest, []).append(f)

duplicate_groups = [group for group in hash_to_files.values() if len(group) > 1]
if duplicate_groups:
    print(f"완전 동일 파일 {len(duplicate_groups)}그룹 발견 -> 각 그룹의 첫 파일만 사용")
    for group in duplicate_groups:
        print(" 중복:", group)
else:
    print("완전 동일 파일 없음")

files = sorted(group[0] for group in hash_to_files.values())
print(f"사용할 파일 {len(files)}개")


완전 동일 파일 없음
사용할 파일 48개


In [4]:
# 모든 파일을 읽어 하나의 DataFrame으로 합치기 (어느 파일에서 왔는지 표시)
df_list = []
for f in files:
    tmp = pd.read_excel(f, dtype={KEY_COLUMN: str})
    tmp["출처파일"] = f
    df_list.append(tmp)

df_all = pd.concat(df_list, ignore_index=True)
print(f"합친 전체 행 수: {len(df_all)}")

# 굿즈 세트 등 실제 도서가 아닌 상품 제외
before = len(df_all)
df_all = df_all[~df_all["상품명"].isin(EXCLUDED_TITLES)].reset_index(drop=True)
print(f"제외된 상품 행 수: {before - len(df_all)}")
print(df_all.head())


합친 전체 행 수: 962
제외된 상품 행 수: 1
   순위           상품코드  ...  분야                                               출처파일
0   1  9788937460586  ...  소설  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
1   2  9788925588735  ...  소설  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
2   3  9791198547514  ...  소설  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
3   4  9791141602376  ...  소설  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
4   5  9788998441012  ...  소설  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....

[5 rows x 14 columns]


In [5]:
# 상품코드(ISBN) 기준 역순위 합산 점수(raw_score)와 중복횟수 계산
raw_score = df_all.groupby(KEY_COLUMN)[RANK_COLUMN].apply(lambda ranks: (1 / ranks).sum()).rename("raw_score")
dup_counts = df_all.groupby(KEY_COLUMN).size().rename("중복횟수")

# 도서 메타데이터는 첫 등장 행 기준으로 사용 (참고용)
df_first = df_all.drop_duplicates(subset=KEY_COLUMN, keep="first").copy()

df_ranking = df_first.merge(raw_score, on=KEY_COLUMN).merge(dup_counts, on=KEY_COLUMN)
print(f"고유 도서 수: {len(df_ranking)}")


고유 도서 수: 87


In [6]:
# 중복횟수(등장 개월 수)의 영향력을 제곱근으로 감쇠시켜 최종 점수 산출
# score = raw_score / sqrt(중복횟수) = 평균순위점수 x sqrt(중복횟수)
df_ranking["score"] = df_ranking["raw_score"] / (df_ranking["중복횟수"] ** 0.5)

df_ranking = df_ranking.sort_values("score", ascending=False).reset_index(drop=True)
df_ranking.insert(0, "final_rank", df_ranking.index + 1)

df_ranking = df_ranking[
    ["final_rank", "score", "raw_score", KEY_COLUMN, "상품명", "인물", "출판사", "중복횟수", RANK_COLUMN, "출처파일"]
]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print(df_ranking.head(20))


    final_rank     score  raw_score           상품코드                     상품명            인물       출판사  중복횟수  순위                                               출처파일
0            1  2.629925  13.665491  9791141602451                      절창           구병모      문학동네    27  20  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (22)....
1            2  2.540150  15.451127  9788925588735               프로젝트 헤일메리         앤디 위어   알에이치코리아    37   2  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
2            3  2.249528  11.688889  9791194530701           괴테는 모든 것을 말했다        스즈키 유이        리프    27   9  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
3            4  2.157886  14.950270  9788936439743                     혼모노           성해나        창비    48  17  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
4            5  2.125211  14.256349  9791199305304                  자몽살구클럽           한로로       어센틱    45   7  ../data/raw/top-list\교보문고_종합_베스트셀러_상품리스트 (14)....
5            6  1.929312  13.366667  978

In [7]:
df_ranking.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH} ({len(df_ranking)}행)")


저장 완료: ../data/processed/book_popularity_baseline.csv (87행)
